In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# OFF baseline: one fixed four-output diagnostic

Current delivery has not run a new GPU experiment. After approval of one L4 24GB-or-higher, 3600-second run, execute top-to-bottom; the GPU cell directly invokes the real runner without another unlock flag.

M8: current manual/fork 8-step BF16 VAE; P8: stock pipeline 8-step BF16 VAE; V8: P8 final latent, freshly checkpoint-loaded FP32 VAE; P50: stock 50-step with that FP32 VAE. Transformer/text encoder stay BF16 throughout. Fixed cube prompt/seed, dimensions and observer. Four videos, 132 actual transformer calls, four VAE decodes; no six-arm intervention, no threshold/amplitude changes, no retry.


## 1. Clone the current authorized source branch


In [ ]:
from pathlib import Path
import subprocess, sys, json, os
REPOSITORY = 'https://github.com/RICHAAARC/SC-SSTW.git'
BRANCH = 'dev/真实视频公共观测/帧差加权质心-固定镜头单主体'
SOURCE = Path('/content/sc_sstw_off_baseline')
if SOURCE.exists():
    raise FileExistsError('Keep prior checkout; use a fresh Colab runtime for this one-run notebook')
subprocess.run(['git', 'clone', '--depth', '1', '--single-branch', '--branch', BRANCH, REPOSITORY, str(SOURCE)], check=True)
SOURCE_COMMIT = subprocess.check_output(['git', '-C', str(SOURCE), 'rev-parse', 'HEAD'], text=True).strip()
print('Source:', SOURCE_COMMIT)


## 2. Dependencies
Only diffusers is exactly pinned, as in the actually used historical G1 installation. Keep the current CUDA PyTorch; record actual versions.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
    'diffusers==0.35.2', 'transformers', 'accelerate', 'ftfy',
    'sentencepiece', 'safetensors', 'huggingface_hub', 'numpy', 'Pillow'], check=True)
subprocess.run(['ffmpeg', '-version'], check=True, stdout=subprocess.DEVNULL)


## 3. Read the fixed four-output plan


In [ ]:
CONFIG = SOURCE / 'configs/stage2_off_baseline.json'
print(json.dumps(json.loads(CONFIG.read_text()), indent=2))
OUTPUT = Path('/content/drive/MyDrive/Video-WM/stage2-off-baseline/run01')
print('Output:', OUTPUT)


## 4. One approved GPU diagnostic
No new GPU execution was performed while preparing this notebook. This cell is the actual run entry: execute only after the single-run approval. The full worker process group is killed at 3600 seconds, and all four output identities/failures are retained. Model loading is included; dependencies and source cloning occur above.


In [ ]:
exit_file = OUTPUT.parent / (OUTPUT.name + '.execution_exit.json')
log_file = OUTPUT.parent / (OUTPUT.name + '.execution.log')
if OUTPUT.exists() or log_file.exists() or exit_file.exists():
    raise FileExistsError('Existing output or execution record is preserved; do not overwrite')
env = dict(os.environ, PYTHONPATH=str(SOURCE), TOKENIZERS_PARALLELISM='false')
completed = subprocess.run([sys.executable, '-m', 'experiments.stage2.run_off_baseline',
    '--config', str(CONFIG), '--output', str(OUTPUT)], cwd=SOURCE, env=env)
print(exit_file.read_text() if exit_file.exists() else 'Launcher failed before recording an execution')
print('Launcher exit:', completed.returncode)
if OUTPUT.exists() and exit_file.exists():
    with (OUTPUT / 'source_commit.txt').open('x') as source_record:
        source_record.write(SOURCE_COMMIT + '\n')


## 5. Inspect the finite diagnostic, not an automatic quality PASS
Review all four first/middle/last previews, contrast and time differences, full observer validity, and raw pairwise comparisons. Differences are conditional on this prompt/seed; no single improvement proves a universal cause.


In [ ]:
result_file = OUTPUT / 'result.json'
print(result_file.read_text() if result_file.exists() else 'Incomplete; inspect failure and execution records')
from IPython.display import display, Image
for output_id in ['M8', 'P8', 'V8', 'P50']:
    preview = OUTPUT / output_id / 'first_middle_last.png'
    print(output_id)
    if preview.exists():
        display(Image(filename=str(preview)))
    else:
        print('No preview; retain missing output and failure reason')
